# RAG

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('..'))
from src.chunkings import chunking_fijo, chunking_oraciones, chunking_parrafos
from src.busqueda_faiss import buscar_en_estrategia
from responder_ia import responder_ia

df = pd.read_csv('../data/5k_resenias.csv', sep=';')
df = df[df['review_text'].notna()]
df["resena_id"] = np.arange(1, len(df) + 1)
print(len(df))
df.head(3)


5089


,business_name,total_reviews,review_rating,review_text,datetime_utc,pais,resena_id
0,Visita guiada por el Museo del Prado y el Pala...,1647,5,"La experiencia fue excelente, desde mucho ante...",07/08/2026,ESP,1
1,Visita guiada por el Museo del Prado y el Pala...,1647,5,Nuestro guía Juan Antonio tenía una gran exper...,07/02/2026,ESP,2
2,Visita guiada por el Museo del Prado y el Pala...,1647,5,"El tour comenzó en el Palacio Real, siguiendo ...",05/30/2026,ESP,3


## Chunking

In [ ]:
df['chunks_fijo'] = df['review_text'].apply(
    lambda x: chunking_fijo(x, tamano_chunk=120, overlap=30)
)
df['chunks_oraciones'] = df['review_text'].apply(
    lambda x: chunking_oraciones(x, oraciones_por_chunk=2, overlap_oraciones=1)
)
df['chunks_parrafos'] = df['review_text'].apply(chunking_parrafos)

estrategias = {
    'fijo': 'chunks_fijo',
    'oraciones': 'chunks_oraciones',
    'parrafos': 'chunks_parrafos',
}

In [6]:
for nombre, col in [
    ('Fijo', 'chunks_fijo'),
    ('Oraciones', 'chunks_oraciones'),
    ('Párrafos', 'chunks_parrafos'),
]:
  todos_los_chunks = [c for lista in df[col] for c in lista]
  tamanos = [len(c) for c in todos_los_chunks]
  print(f'Estrategia {nombre}:')
  print(f'  Total chunks generados: {len(todos_los_chunks)}')
  print(f'  Tamaño promedio: {np.mean(tamanos):.0f} caracteres')
  print(f'  Min/Max: {min(tamanos)}/{max(tamanos)} caracteres\n')

Estrategia Fijo:
  Total chunks generados: 10800
  Tamaño promedio: 83 caracteres
  Min/Max: 1/120 caracteres

Estrategia Oraciones:
  Total chunks generados: 10861
  Tamaño promedio: 103 caracteres
  Min/Max: 1/1483 caracteres

Estrategia Párrafos:
  Total chunks generados: 5089
  Tamaño promedio: 146 caracteres
  Min/Max: 3/3203 caracteres



## Embeddings

In [2]:
modelo_e5 = SentenceTransformer('intfloat/multilingual-e5-small')

In [ ]:
dict_embeddings_guardar = {}


for nombre_est, columna_chunk in estrategias.items():
  registros_chunks = []
  for _, row in df.iterrows():
    for idx_chunk, texto_chunk in enumerate(row[columna_chunk]):
      registros_chunks.append({
          'resena_id': row['resena_id'],
          'chunk_index': idx_chunk,
          'texto': texto_chunk,
      })

  df_chunks = pd.DataFrame(registros_chunks)


  textos_e5 = [f'passage: {t}' for t in df_chunks['texto']]

  embeddings = modelo_e5.encode(textos_e5, show_progress_bar=True)

  df_chunks.to_csv(f'metadata_chunks_{nombre_est}.csv', index=False)
  np.save(f'embeddings_{nombre_est}.npy', embeddings)

In [3]:
embeddings_oraciones_cargados = np.load('embeddings_oraciones.npy')
metadata_oraciones_cargada = pd.read_csv('metadata_chunks_oraciones.csv')

print(
    '\nEmbeddings:'
    f' Formato: {embeddings_oraciones_cargados.shape}'
)


Embeddings: Formato: (10861, 384)


## Busqueda Semantica con FAISS

In [4]:
pregunta_prueba = '¿Las atracciones estan limpias?'
estrategias = ['fijo', 'oraciones', 'parrafos']

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Las atracciones estan limpias?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 1628 | Score: 0.7685
    Texto: 'as, pero no muy limpias, sin embargo es de entender por la vegetación del lugar. En cuanto a atracciones hay 3, me encan'
[2] Reseña ID: 2364 | Score: 0.7566
    Texto: 'ho de que a pesar de que en general el mantenimiento del lugar está bien, algunas de las atracciones sí les falta manten'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 1573 | Score: 0.7678
    Texto: 'Al menos limpieza, es una lastima ya que es bonito, pero especialmente el area de juegos de niños esta sumamente sucia y es desagradable.'
[2] Reseña ID: 2101 | Score: 0.7665
    Texto: 'Muchas atracciones, muy amplio, muy limpio.'

--- ESTRATEGIA: PARRAFOS ---
[1] Reseña ID: 2353 | Score: 0.7547
    Texto: 'Un lugar maravilloso para los niños. Hay que llegar temprano para tener tiempo de recorrer todo. Sin embargo el mantenimiento deja mucho que desear. Las atracciones están cada vez más deterio

In [ ]:
pregunta_prueba = '¿Cuales son los peores tours de Civitatis?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Cuales son los mejores tours de Civitatis?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 2677 | Score: 0.7430
    Texto: 'La peor actividad que hemos realizado con Civitatis. Eramos 5 turistas. La guía (una tal Cecilia parece que ponía en su'
[2] Reseña ID: 3345 | Score: 0.7345
    Texto: 'da en la compra de entradas al Museo. Civitatis fue la mejor opción. Totalmente aconsejable.'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 3345 | Score: 0.7262
    Texto: 'Después de todas las opciones de búsqueda en la compra de entradas al Museo. Civitatis fue la mejor opción.'
[2] Reseña ID: 1118 | Score: 0.7226
    Texto: 'Es la primera vez que dejo una mala reseña en Civitatis. Hace años que hago tours con la empresa, por varios países y siempre encuentro algo positivo, podrán ver mis reseñas, pero está vez, las siento no encuentro nada.'

--- ESTRATEGIA: PARRAFOS ---
[1] Reseña ID: 3313 | Score: 0.7593
    Texto: 'Civitatis solo hace de intermediarios y se lava las manos. El 

In [15]:
pregunta_prueba = '¿Que lugares son los más económicos?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Que lugares son los más económicos?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 2361 | Score: 0.7369
    Texto: 'l Museo. Es económico y los más pequeños la pueden pasar súper bien en las zonas de juego. Siempre hay actividades espec'
[2] Reseña ID: 2366 | Score: 0.7223
    Texto: 'queña donde puedes comprar algo para comer (incluso con plato del día) a precios muy asequibles.'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 1842 | Score: 0.7250
    Texto: 'Son bastante económicas y accesibles. Para los niños, hay muchas actividades como toboganes y piscinas donde pueden disfrutar.'
[2] Reseña ID: 2263 | Score: 0.7209
    Texto: 'Les diré lo que me dijo mi sobrino "Aquí todo es mágico tío?" De los mejores lugares donde pueden ir los niños a divertirse y a aprender a precios absurdos'

--- ESTRATEGIA: PARRAFOS ---
[1] Reseña ID: 2263 | Score: 0.7209
    Texto: 'Les diré lo que me dijo mi sobrino "Aquí todo es mágico tío?" De los mejores lugares donde pueden ir los niño

In [18]:
pregunta_prueba = '¿Es posible ver el volcan cuando hay neblina?'

print(f"BÚSQUEDA SEMÁNTICA: '{pregunta_prueba}'\n" + '=' * 60)

for est in estrategias:
  print(f'\n--- ESTRATEGIA: {est.upper()} ---')
  resultados = buscar_en_estrategia(
      pregunta_prueba, est, modelo_e5, top_k=2
  )

  for i, r in enumerate(resultados):
    print(f"[{i+1}] Reseña ID: {r['resena_id']} | Score: {r['score']:.4f}")
    print(f"    Texto: '{r['chunk']}'")

BÚSQUEDA SEMÁNTICA: '¿Es posible ver el volcan cuando hay neblina?'

--- ESTRATEGIA: FIJO ---
[1] Reseña ID: 1231 | Score: 0.7179
    Texto: 'súper vale la pena ver esta maravilla de la naturaleza. El volcán se ve bien como a las 6 de la tarde aprox.'
[2] Reseña ID: 1243 | Score: 0.7083
    Texto: 'isita a parte del contacto con la naturaleza que es increíble son las vistas al Volcán que no siempre pueden lograrse po'

--- ESTRATEGIA: ORACIONES ---
[1] Reseña ID: 1334 | Score: 0.7235
    Texto: 'Volcán perfectamente cónico, centro de un hermoso parque nacional. Tiene hermosas vistas desde la base o la laguna, pero hay que tener paciencia para verlo completo porque siempre hay nubes.'
[2] Reseña ID: 1231 | Score: 0.7100
    Texto: 'Sólo hay dos estaciones verano (Ene-junio) e invierno(julio-dic), yo fui en invierno y el clima es muy cambiante sale el sol en cuestión de minutos de nubla y llueve, sin embrago, súper vale la pena ver esta maravilla de la naturaleza. El volcán se ve bien co

## Generador

In [ ]:
pregunta_usuario = "¿Angel es un buen guia?"
estrategia_elegida = "oraciones"

resultados_faiss = buscar_en_estrategia(
    pregunta=pregunta_usuario,
    nombre_estrategia=estrategia_elegida,
    modelo=modelo_e5,
    top_k=3,
)

contexto_recuperado = "\n\n".join([r["chunk"] for r in resultados_faiss])

print("--- CONTEXTO RECUPERADO DE LA BASE DE DATOS ---")
print(contexto_recuperado)
print("------------------------------------------------\n")

modelos_locales = ['qwen2.5:7b', 'qwen2.5:3b', 'gemma2:2b', 'llama3.2:3b'] 


# ollama run qwen2.5:7b
respuesta_final = responder_ia(
    pregunta=pregunta_usuario, contexto=contexto_recuperado, modelo_ollama=modelos_locales[3] # 3
)

print(f"PREGUNTA: {pregunta_usuario}\n")
print(f"RESPUESTA: \n{respuesta_final}")

--- CONTEXTO RECUPERADO DE LA BASE DE DATOS ---
El guía, Angel, es una maravilla, para la actividad tan densa que tiene que realizar es capaz de mantener el interés del grupo además de su propio entusiasmo por enseñar. Solo por poner una pega, estaría bien dedicar la mañana a un monumento y la tarde a otro dejando un descanso en medio para comer y descansar un rato.

Ángel es un guía fantástico. Atento, agradable y con muchos conocimientos.

Angel es un guía simpático pero con muy poco conocimiento histórico. Habla muy poco de historia y mucho de cosas irrelevantes y repetitivas.
------------------------------------------------

PREGUNTA: ¿Angel es un buen guia?

RESPUESTA: 
Sí, Angel es considerado un buen guía por muchos. Según las reseñas, Angel mantiene el interés del grupo y demuestra entusiasmo en su trabajo, lo que contribuye a una experiencia positiva. Sin embargo, también se menciona que podría mejorar al dedicar más tiempo a monumentos específicos y incluir un descanso. Algun

In [ ]:
responder_ia(pregunta=pregunta_usuario, contexto=contexto_recuperado, modelo='gemini-3.5-flash-lite' gemini_api_key="____")

'Según las reseñas proporcionadas, las opiniones sobre el guía Ángel están divididas:\n\n* Para algunos es una maravilla capaz de mantener el interés del grupo y transmitir su entusiasmo, además de ser descrito como un guía fantástico, atento, agradable y con muchos conocimientos.\n* Para otro usuario, en cambio, es simpático pero considera que tiene muy poco conocimiento histórico, ya que habla muy poco de historia y se centra en cosas irrelevantes y repetitivas.'